# CRYSTALS-Kyber with Complete Mathematical Walkthrough

**Post-Quantum Key Encapsulation Mechanism | ML-KEM / FIPS 203**

This notebook teaches the **complete mathematics** of the Kyber crystal algorithm.
Every concept is explained first, then immediately implemented in Python.

**Toy parameters** that have identical structure to FIPS 203, just smaller numbers:

| Symbol | Value | Meaning |
|--------|-------|---------|
| q | 17 | Prime modulus, all coefficients live in {0 … 16} |
| n | 8 | Polynomial degree, ring modulus is X⁸ + 1 |
| k | 2 | Module dimension, A is 2×2, vectors have 2 polynomials |
| η | 2 | CBD noise bound, secret coefficients in {−2,−1,0,1,2} |

**How to use this notebook:**
1. Run **Module 0 Global Setup** first. It defines all shared constants.
2. Work through each module in order. Every cell builds on the previous.
3. All arithmetic is mod **q = 17** unless stated otherwise.

> **All numbers in this notebook are pre-verified against the CRYSTALS-Kyber reference document.**

## Module 0: The Global Setup

> **Run this cell first.** Every other cell depends on these constants.
> Do not modify the values below. They are the verified ground truth for all exercises.

The constants are split into three groups:
- **Parameters** which are the mathematical settings of this Kyber instance
- **Key Generation values** secret key s, error e, NTT roots, public key t
- **Encapsulation and Decapsulation values** ciphertext u, v and recovery w

In [ ]:
# Setup: toy parameters and reference values (run this cell first)

# Toy parameters:
Q     = 17   # prime modulus
N     = 8    # polynomial degree
K     = 2    # module dimension
ETA   = 2    # CBD noise parameter eta
N_INV = 15   # multiplicative inverse of n:  8 × 15 = 120 ≡ 1 mod 17

# NTT roots
# roots[k]     = ω^k  mod 17   where ω = 9
# inv_roots[k] = (ω⁻¹)^k mod 17  where ω⁻¹ = 2
ROOTS     = [1, 9, 13, 15, 16, 8, 4, 2]
INV_ROOTS = [1, 2,  4,  8, 16, 15, 13, 9]

# Matrix A (NTT domain, generated from public seed ρ)
A = [
    [[2,11,8,1,15,3,3,6],  [6,6,9,9,12,12,15,15]],
    [[7,14,3,10,5,12,1,8], [3,7,2,9,14,4,11,6  ]]
]

# Secret key s and error e (CBD output, normal domain)
S0 = [1, 0, 0, 1, 0, 0, 0, 1]   # from bytes [0x01, 0x10, 0x00, 0x10]
S1 = [0, 1, 0, 0, 1, 0, 0, 0]   # from bytes [0x10, 0x00, 0x01, 0x00]
E0 = [1, 0, 0, 0, 0, 0, 0, 0]   # from bytes [0x01, 0x00, 0x00, 0x00]
E1 = [0, 0, 1, 0, 0, 0, 0, 0]   # from bytes [0x00, 0x01, 0x00, 0x00]

# NTT domain forms of s and e
S0_NTT = [3,  1,  9,  1, 16,  1, 10,  1]
S1_NTT = [2,  8, 14, 14,  0,  7,  5,  1]
E0_NTT = [1,  1,  1,  1,  1,  1,  1,  1]
E1_NTT = [1, 13, 16,  4,  1, 13, 16,  4]

# Public key t (NTT domain)
T0 = [2,  9, 12, 9,  3,  3,  4,  5]
T1 = [11, 15,  3, 4, 13,  2, 13,  1]

# Encapsulation: fresh noise and message
R0    = [1, 0, 1, 0, 0, 1, 0, 0]
R1    = [0, 1, 0, 0, 1, 0, 1, 0]
E1_0  = [0, 0, 0, 1, 0, 0, 0, 0]
E1_1  = [0, 0, 0, 0, 0, 1, 0, 0]
E2    = [0, 0, 0, 0, 1, 0, 0, 0]
R0_NTT = [3,  5, 13,  7,  1,  6,  4,  3]
R1_NTT = [3, 12, 13, 10,  1, 11,  4, 14]
MSG   = [1, 0, 1, 0, 1, 0, 1, 0]
M_HAT = [8, 0, 8, 0, 8, 0, 8, 0]   # message encoded: bit→8 if 1, bit→0 if 0

# Ciphertext
U0 = [0, 15,  3, 12,  9, 16,  3,  4]
U1 = [8,  7,  5,  8, 16, 12,  6,  0]
V  = [6,  6,  6,  9, 14, 12,  9, 10]

# Decapsulation intermediate values
U0_NTT = [11,  0, 11, 14,  2, 16, 12,  2]
U1_NTT = [11,  3,  3,  2,  8,  6,  6,  8]
STU_N  = [13,  7, 15,  9,  5, 11,  2, 10]
W      = [10, 16,  8,  0,  9,  1,  7,  0]
DECODED= [ 1,  0,  1,  0,  1,  0,  1,  0]

print(f"Setup complete.  Q={Q}  N={N}  K={K}  ETA={ETA}  N_INV={N_INV}")
print(f"Forward roots:   {ROOTS}")
print(f"Inverse roots:   {INV_ROOTS}")

## Module 1: The Mathematical Foundations

### 1.1 The Ring R_q = Z₁₇[X] / (X⁸ + 1)

All Kyber arithmetic lives in this ring:
- **Every polynomial has at most 8 terms** in other words coefficients on X⁰ through X⁷
- **All coefficients are integers mod q = 17** and they live in {0, 1, …, 16}
- **Negacyclic reduction:** when multiplication produces X^k with k ≥ 8, we use the identity
 `X⁸ ≡ −1 mod (X⁸+1)`, giving:
 `X⁸ → −1`, `X⁹ → −X`, `X¹⁰ → −X²`, …, `X¹⁵ → −X⁷`

**Example:** a coefficient on X¹⁰ wraps as `−1 × (coefficient on X²) mod 17`

### 1.2 Finding the primitive root g

We need a generator g of Z*₁₇ is an element whose powers produce all 16 nonzero residues mod 17.

**Test:** g must have **order 16**, meaning g^16 ≡ 1 mod 17 AND g^k ≠ 1 for all 1 ≤ k < 16.
Equivalently (since 16 = 2⁴, only prime factor is 2): check that g^8 ≠ 1 mod 17.

- g = 2 **fails**: 2⁸ mod 17 = 1 → order is only 8, not 16
- g = 3 **works**: 3⁸ mod 17 = 16 = −1 ≠ 1 → order is 16 ✓

### 1.3 Deriving ψ, ω, and the roots tables

From g = 3 we derive everything needed for the NTT:

```
ψ = g^((q−1)/(2n)) mod q = 3^((17−1)/(2×8)) = 3¹ mod 17 = 3
 Check: ψ^16 = 1 mod 17 ✓ (primitive 2n-th root of unity)
 Check: ψ^8 = 16 = −1 mod 17 ✓ (must not equal 1)

ω = ψ² mod 17 = 9
 Check: ω^8 = 1 mod 17 ✓ (primitive n-th root of unity)

ω⁻¹ = 2 because 9 × 2 = 18 ≡ 1 mod 17
n⁻¹ = 15 because 8 × 15 = 120 ≡ 1 mod 17

roots[k] = ω^k mod 17 = [1, 9, 13, 15, 16, 8, 4, 2]
inv_roots[k] = (ω⁻¹)^k mod 17 = [1, 2, 4, 8, 16, 15, 13, 9]
```

**Verify:** `roots[k] × inv_roots[k] ≡ 1 mod 17` for all k

### Exercise 1: Implement `modpow` and verify g, ψ, ω

Implement fast modular exponentiation using the **square-and-multiply** method:

```
result = 1
while exp > 0:
 if exp is odd: result = result × base mod mod
 base = base² mod mod
 exp = exp // 2
```

Then use it to verify the derivation chain g → ψ → ω → roots tables.

In [ ]:
def modpow(base: int, exp: int, mod: int) -> int:
    """Fast modular exponentiation (square-and-multiply).

    Algorithm:
        result = 1
        while exp > 0:
            if exp is odd:  result = result * base % mod
            base = base^2 % mod
            exp  = exp >> 1   (bit-shift)
    """
    result = 1
    b      = base % mod
    while exp > 0:
        if exp & 1:
            # TODO: multiply result by b, reduce mod mod
            result = ...
        # TODO: square b, reduce mod mod
        b   = ...
        # TODO: right-shift exp by 1  (exp >>= 1)
        exp = ...
    return result

print("Does g = 2 work?")
print(f"2^8  mod 17 = {modpow(2,  8, Q)}   (need ≠ 1 for full order 16)")
print(f"2^16 mod 17 = {modpow(2, 16, Q)}   (order is 8, not 16 → REJECTED ✗)")
print("\nDoes g = 3 work?")
print("k    3^k mod 17")
elements = set()
for k in range(1, 17):
    v = modpow(3, k, Q)
    elements.add(v)
    mark = "  ← identity, order = 16 confirmed ✓" if v == 1 else ""
    print(f"  {k:2d}     {v:2d}{mark}")
print(f"\nGenerated {len(elements)} distinct values → g = 3 ACCEPTED ✓")
psi       = modpow(3, (Q-1)//(2*N), Q)
omega     = modpow(psi, 2, Q)
omega_inv = modpow(omega, Q-2, Q)
n_inv_v   = modpow(N, Q-2, Q)
print(f"\nψ={psi}  ω={omega}  ω⁻¹={omega_inv}  n⁻¹={n_inv_v}")
fwd = [modpow(omega,     k, Q) for k in range(N)]
inv = [modpow(omega_inv, k, Q) for k in range(N)]
print(f"Forward roots:  {fwd}")
print(f"Inverse roots:  {inv}")
assert fwd == ROOTS and inv == INV_ROOTS
print("\n✓ Both roots tables match the global constants.")


## Module 2 CBD Sampling: Generating s and e

### 2.1 Why small coefficients?

Kyber's security depends on the secret key **s** and error **e** having **small** coefficients.
The noise term in decapsulation must satisfy `|noise| < q/4 = 4.25` for correct decryption.
CBD with η=2 guarantees coefficients in {−2, −1, 0, +1, +2}, which is exactly small enough.

### 2.2 The CBD algorithm (η = 2)

Input: a byte stream from `SHAKE256(σ ‖ counter)`, read **LSB-first** (bit 0 of byte first).

For every 4 bits `(b₀, b₁, b₂, b₃)`:
```
coefficient = (b₀ + b₁) − (b₂ + b₃)
```
This has the **centered binomial distribution** with parameter η=2:
values −2, −1, 0, +1, +2 with probabilities 1/16, 4/16, 6/16, 4/16, 1/16.

**Negative coefficients** are stored as their positive representative mod q:
`−1 → 16`, `−2 → 15`

**One byte → 2 coefficients. Four bytes → one polynomial (n=8 coefficients).**

### 2.3 Full trace from the reference document

```
s[0] input bytes: [0x01, 0x10, 0x00, 0x10]

Byte 0x01 = 00000001 (LSB-first: 1,0,0,0,0,0,0,0)
 coeff[0]: (b0+b1)−(b2+b3) = (1+0)−(0+0) = +1 → stored: 1
 coeff[1]: (b4+b5)−(b6+b7) = (0+0)−(0+0) = 0 → stored: 0

Byte 0x10 = 00010000 (LSB-first: 0,0,0,0,1,0,0,0)
 coeff[2]: (0+0)−(0+0) = 0 → stored: 0
 coeff[3]: (1+0)−(0+0) = +1 → stored: 1

Byte 0x00 → coeff[4]=0, coeff[5]=0
Byte 0x10 → coeff[6]=0, coeff[7]=+1

s[0] = [1, 0, 0, 1, 0, 0, 0, 1] ✓
```

### Exercise 2: Implement `cbd_sample`

Convert a list of bytes to polynomial coefficients using the CBD algorithm.

In [ ]:
def cbd_sample(byte_list: list, q: int) -> list:
    """CBD η=2: convert bytes to polynomial coefficients.

    For each byte:
      1. bits = [(byte_val >> i) & 1  for i in range(8)]    ← LSB first
      2. For each group of 4 bits (pair = 0 or 1):
             b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
             coeff = (b0 + b1) - (b2 + b3)     ← centered binomial
             store coeff % q
    """
    coefficients = []
    for byte_val in byte_list:
        bits = [(byte_val >> i) & 1 for i in range(8)]
        for pair in range(2):
            b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
            # TODO: compute coeff = (b0 + b1) - (b2 + b3)
            coeff = ...
            # TODO: append coeff % q  to coefficients
            coefficients.append(...)
    return coefficients

print("CBD for s[0] bytes [0x01, 0x10, 0x00, 0x10]\n")
for idx, byte_val in enumerate([0x01, 0x10, 0x00, 0x10]):
    bits = [(byte_val >> i) & 1 for i in range(8)]
    print(f"  Byte 0x{byte_val:02X}  LSB-first: {bits}")
    for pair in range(2):
        b0,b1,b2,b3 = bits[pair*4:pair*4+4]
        raw = (b0+b1)-(b2+b3); stored = raw%Q
        print(f"    coeff[{idx*2+pair}]: ({b0}+{b1})-({b2}+{b3})={raw:+d}  stored:{stored}")
    print()
s0=cbd_sample([0x01,0x10,0x00,0x10],Q); s1=cbd_sample([0x10,0x00,0x01,0x00],Q)
e0=cbd_sample([0x01,0x00,0x00,0x00],Q); e1=cbd_sample([0x00,0x01,0x00,0x00],Q)
print(f"s[0]={s0}  expected:{S0}")
print(f"s[1]={s1}  expected:{S1}")
print(f"e[0]={e0}  expected:{E0}")
print(f"e[1]={e1}  expected:{E1}")
assert s0==S0 and s1==S1 and e0==E0 and e1==E1
print("\n✓ All CBD outputs match the global constants.")


## Module 3: Forward NTT (Cooley-Tukey)

### 3.1 Why NTT?

Polynomial multiplication in R_q naively takes **O(n²)** operations.
The **Number Theoretic Transform** converts polynomials to a domain where
multiplication is **pointwise** (coefficient × coefficient), reducing complexity to **O(n log n)**.

### 3.2 Bit-reversal permutation

Before the butterfly layers, the input array is **permuted** by reversing the binary representation of each index.

For n=8 (3-bit indices):

| Index | Binary | Reversed | New position |
|-------|--------|----------|--------------|
| 0 | 000 | 000 | 0 |
| 1 | 001 | 100 | 4 |
| 2 | 010 | 010 | 2 |
| 3 | 011 | 110 | 6 |
| 4 | 100 | 001 | 1 |
| 5 | 101 | 101 | 5 |
| 6 | 110 | 011 | 3 |
| 7 | 111 | 111 | 7 |

Pairs (1,4) and (3,6) swap. Positions {0,2,5,7} stay.

### 3.3 Butterfly operation (Cooley-Tukey)

Each butterfly takes two values **u** and **v**, and a **twiddle factor w**:
```
t = v × w mod q
new_u = (u + t) mod q
new_v = (u − t) mod q
```

### 3.4 Three butterfly layers for n=8

```
Layer 1: len=2, step=4 (4 blocks of 2 elements, root = roots[0] = 1)
Layer 2: len=4, step=2 (2 blocks of 4 elements)
Layer 3: len=8, step=1 (1 block of 8 elements)
```

In each layer: `step = n // len`.
For block starting at `bs`, butterfly positions `j` and `j + len//2`:
`w = roots[j × step]`

### 3.5 Full trace for s[0] = [1,0,0,1,0,0,0,1]

```
After bit-reversal: [1, 0, 0, 0, 0, 0, 1, 1]

Layer 1 (w=1 for all):
 [0,1]: u=1,v=0,w=1 → t=0 → [1, 1]
 [2,3]: u=0,v=0 → t=0 → [0, 0]
 [4,5]: u=0,v=0 → t=0 → [0, 0]
 [6,7]: u=1,v=1,w=1 → t=1 → [2, 0]
After layer 1: [1, 1, 0, 0, 0, 0, 2, 0]

Layer 2 (roots[0]=1, roots[2]=13):
 [0,2]: w=1 t=0 → [1, 1]
 [1,3]: w=13 t=0 → [1, 1]
 [4,6]: w=1 t=2 → [2, 15] (0−2 mod 17 = 15)
 [5,7]: w=13 t=0 → [0, 0]
After layer 2: [1, 1, 1, 1, 2, 0, 15, 0]

Layer 3 (roots[0..3] = 1, 9, 13, 15):
 [0,4]: w=1 t=2 → [3, 16]
 [1,5]: w=9 t=0 → [1, 1]
 [2,6]: w=13 t=15×13 mod 17 = 8 → [9, 10]
 [3,7]: w=15 t=0 → [1, 1]
After layer 3: [3, 1, 9, 1, 16, 1, 10, 1] ← NTT(s[0])
```

### Exercise 3: Implement `ntt`

In [ ]:
import math

def bit_reverse_permute(a: list) -> list:
    """Return a new list with elements reordered by bit-reversed indices."""
    n    = len(a)
    bits = int(math.log2(n))
    r    = list(a)
    for i in range(n):
        rev = int(f'{i:0{bits}b}'[::-1], 2)
        if rev > i:
            r[i], r[rev] = r[rev], r[i]
    return r

def ntt(poly: list, roots: list, q: int) -> list:
    """Forward NTT using the Cooley-Tukey iterative method.

    Steps:
      1. a = bit_reverse_permute(poly)
      2. length starts at 2, doubles each round (2 → 4 → 8):
             step = n // length
             for each block of size `length` starting at bs:
                 for j in range(length // 2):
                     f = bs + j
                     s = bs + j + length // 2
                     w = roots[j * step]
                     t = a[s] * w % q
                     a[f], a[s] = (a[f]+t)%q, (a[f]-t)%q
    """
    n = len(poly)
    a = bit_reverse_permute(poly)
    length = 2
    while length <= n:
        # TODO: step = n // length
        step = ...
        for bs in range(0, n, length):
            for j in range(length // 2):
                f = bs + j
                # TODO: s = bs + j + length // 2
                s = ...
                # TODO: w = roots[j * step]
                w = ...
                t = a[s] * w % q
                # TODO: butterfly, set a[f], a[s] = (a[f]+t)%q, (a[f]-t)%q
                #       Important: compute both from the OLD a[f] value!
                a[f], a[s] = ...
        # TODO: length <<= 1  (double it to move to next layer)
        length = ...
    return a

print("=== Full NTT trace for s[0] ===\n")
a_trace = bit_reverse_permute(list(S0))
print(f"After bit-reversal: {a_trace}\n")
length = 2; layer = 1
while length <= N:
    step = N // length
    print(f"Layer {layer}  (len={length}, step={step}):")
    for bs in range(0, N, length):
        for j in range(length // 2):
            f,s = bs+j, bs+j+length//2
            w = ROOTS[j*step]; t = a_trace[s]*w%Q; u_old=a_trace[f]
            a_trace[f]=(a_trace[f]+t)%Q; a_trace[s]=(u_old-t)%Q
            print(f"  [{f},{s}] w={w} t={t} → [{f}]={a_trace[f]} [{s}]={a_trace[s]}")
    print(f"  After layer {layer}: {a_trace}\n")
    length<<=1; layer+=1
print(f"NTT(s[0])={a_trace}  expected={S0_NTT}")
assert ntt(S0,ROOTS,Q)==S0_NTT and ntt(S1,ROOTS,Q)==S1_NTT
assert ntt(E0,ROOTS,Q)==E0_NTT and ntt(E1,ROOTS,Q)==E1_NTT
print("\n✓ All NTT results match the global constants.")


## Module 4: Inverse NTT

### 4.1 Structure

The INTT uses the **same butterfly structure** as the forward NTT, with two differences:

1. Pass `inv_roots` instead of `roots`
2. After all butterfly layers: **multiply every coefficient by n⁻¹ = 15**

```
INTT(a) = scale(forward_butterfly(bit_reverse(a), inv_roots), n_inv)
```

This is why we define a single `ntt()` function and call it twice.

### 4.2 Round-trip trace for NTT(s[0]) = [3,1,9,1,16,1,10,1]

```
After bit-reversal: [3, 16, 9, 10, 1, 1, 1, 1]

Layer 1 (inv_roots[0]=1):
 [0,1]: t=16 → [2, 4] (3+16=19≡2; 3-16=-13≡4)
 [2,3]: t=10 → [2, 16] (9+10=19≡2; 9-10=-1≡16)
 [4,5]: t=1 → [2, 0]
 [6,7]: t=1 → [2, 0]
After layer 1: [2, 4, 2, 16, 2, 0, 2, 0]

Layer 2 (inv_roots[0]=1, inv_roots[2]=4):
 [0,2] w=1: t=2 → [4, 0]
 [1,3] w=4: t=16×4=64≡13 → 4+13=17≡0, 4-13=-9≡8 → [0, 8]
 [4,6] w=1: t=2 → [4, 0]
 [5,7] w=4: t=0 → [0, 0]
After layer 2: [4, 0, 0, 8, 4, 0, 0, 0]

Layer 3 (inv_roots[0..3] = 1, 2, 4, 8):
 [0,4] w=1: t=4 → [8, 0]
 [1,5] w=2: t=0 → [0, 0]
 [2,6] w=4: t=0 → [0, 0]
 [3,7] w=8: t=0 → [8, 8]
Before scaling: [8, 0, 0, 8, 0, 0, 0, 8]

Multiply by n⁻¹ = 15: [8×15, 0, 0, 8×15, 0, …] mod 17
 = [120 mod 17, 0, …] = [1, 0, 0, 1, 0, 0, 0, 1]
```

### Exercise 4: Implement `intt`

In [ ]:
def intt(poly: list, inv_roots: list, q: int, n_inv: int) -> list:
    """Inverse NTT.

    Steps:
      1. Call ntt(poly, inv_roots, q)   ← same butterfly, different roots
      2. Multiply every output coefficient by n_inv % q
    """
    # TODO: call ntt with inv_roots (not roots!)
    result = ...
    # TODO: return [x * n_inv % q  for x in result]
    return [... for x in result]

print(f"INTT(NTT(s[0])) = {intt(S0_NTT, INV_ROOTS, Q, N_INV)}")
print(f"Expected s[0]   = {S0}")
assert intt(S0_NTT,INV_ROOTS,Q,N_INV)==S0 and intt(S1_NTT,INV_ROOTS,Q,N_INV)==S1
assert intt(E0_NTT,INV_ROOTS,Q,N_INV)==E0 and intt(E1_NTT,INV_ROOTS,Q,N_INV)==E1
print("\n✓ All INTT round-trips match the global constants.")


## Module 5: Public Key: t̂ = Â·ŝ + ê

### 5.1 Why all computation stays in NTT domain

Once s and e are transformed to NTT domain (ŝ, ê), and A is also in NTT domain (Â),
the public key computation requires only **pointwise operations**, so no INTT is needed here.

```
t̂[0] = Â[0][0] ⊙ ŝ[0] + Â[0][1] ⊙ ŝ[1] + ê[0]
t̂[1] = Â[1][0] ⊙ ŝ[0] + Â[1][1] ⊙ ŝ[1] + ê[1]
```

`⊙` = **pointwise multiply** mod q (coefficient i × coefficient i)
`+` = **pointwise add** mod q

### 5.2 Worked trace for t̂[0], coefficient i=0

```
Â[0][0][0] = 2, ŝ[0][0] = 3 → 2×3 = 6
Â[0][1][0] = 6, ŝ[1][0] = 2 → 6×2 = 12
ê[0][0] = 1

t̂[0][0] = 6 + 12 + 1 = 19 mod 17 = 2 ✓
```

All 8 coefficients of t̂[0]:

| i | Â[0][0]×ŝ[0] | Â[0][1]×ŝ[1] | +ê[0] | t̂[0][i] |
|---|--------------|--------------|-------|----------|
| 0 | 2×3=6 | 6×2=12 | 1 | (6+12+1) mod 17 = **2** |
| 1 | 11×1=11 | 6×8=48→14 | 1 | (11+14+1) mod 17 = **9** |
| 2 | 8×9=72→4 | 9×14=126→7 | 1 | (4+7+1) mod 17 = **12** |
| 3 | 1×1=1 | 9×14=7 | 1 | (1+7+1) mod 17 = **9** |
| 4 | 15×16=240→2 | 12×0=0 | 1 | (2+0+1) mod 17 = **3** |
| 5 | 3×1=3 | 12×7=84→16 | 1 | (3+16+1) mod 17 = **3** |
| 6 | 3×10=30→13 | 15×5=75→7 | 1 | (13+7+1) mod 17 = **4** |
| 7 | 6×1=6 | 15×1=15 | 1 | (6+15+1) mod 17 = **5** |

`t̂[0] = [2, 9, 12, 9, 3, 3, 4, 5]` `t̂[1] = [11, 15, 3, 4, 13, 2, 13, 1]`

### Exercise 5: Implement `keygen_t`

In [ ]:
def pointwise_mul(a: list, b: list, q: int) -> list:
    """Pointwise multiply: return [a[i]*b[i] % q  for each i]."""
    # TODO: return [x * y % q  for x, y in zip(a, b)]
    return [... for x, y in zip(a, b)]

def pointwise_add(a: list, b: list, q: int) -> list:
    """Pointwise add: return [(a[i]+b[i]) % q  for each i]."""
    # TODO: return [(x + y) % q  for x, y in zip(a, b)]
    return [... for x, y in zip(a, b)]

def matvec_ntt(M, v_ntt, q):
    """Matrix-vector NTT multiply (provided, do not modify)."""
    k = len(M); n = len(M[0][0])
    result = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            for i in range(n):
                acc[i] = (acc[i] + M[row][col][i] * v_ntt[col][i]) % q
        result.append(acc)
    return result

def keygen_t(A_hat, s_ntt, e_ntt, q):
    """Compute t̂ = Â·ŝ + ê in NTT domain.

    For each row:
        acc = [0]*n
        for each col:
            prod = A_hat[row][col] ⊙ s_ntt[col]   (pointwise_mul)
            acc  = acc + prod                       (pointwise_add)
        t̂[row] = acc + e_ntt[row]                  (pointwise_add)
    """
    k = len(A_hat); n = len(A_hat[0][0])
    t_hat = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            # TODO: prod = pointwise_mul(A_hat[row][col], s_ntt[col], q)
            prod = ...
            # TODO: acc  = pointwise_add(acc, prod, q)
            acc  = ...
        # TODO: t_hat.append( pointwise_add(acc, e_ntt[row], q) )
        t_hat.append(...)
    return t_hat

t_hat = keygen_t(A, [S0_NTT,S1_NTT], [E0_NTT,E1_NTT], Q)
print(f"t̂[0]={t_hat[0]}  expected {T0}")
print(f"t̂[1]={t_hat[1]}  expected {T1}")
assert t_hat[0]==T0 and t_hat[1]==T1
print("\n✓ Public key t̂ matches the global constants.")


## Module 6: Encapsulation

### 6.1 Overview

Bob holds the public key `pk = (t̂, ρ)`. He chooses a message `m` and fresh noise `r, e1, e2`,
and produces ciphertext `(u, v)` that only Alice (with secret key ŝ) can decrypt.

```
u = INTT(Â^T · r̂) + e1
v = INTT(t̂^T · r̂) + e2 + m̂
```

**Transpose:** `Â^T[i][j] = Â[j][i]`, where rows and columns of A are swapped.

**Message encoding:** each bit of m is scaled to the midpoint of Z_q:
```
bit 0 → 0 (near zero)
bit 1 → ⌊q/2⌋ = 8 (near the modulus midpoint)
```
This maximises the distance from a wrong decoding.

### 6.2 Why the transpose?

The algebraic cancellation in decapsulation requires:

```
v − ŝ^T · u = m̂ + small noise terms
```

For this to work, Bob must multiply by `Â^T` (not `Â`) when computing `u`.
This way the `A·s` term inside `t` cancels with the `A^T·r` term inside `u`.

### 6.3 Computing u, full trace for u[0]

```
Â^T row 0 = [Â[0][0], Â[1][0]] (column 0 of A, not row 0)

Â[0][0] ⊙ r̂[0]: [2×3, 11×5, 8×13, 1×7, 15×1, 3×6, 3×4, 6×3]
 = [6, 55→4, 104→2, 7, 15, 18→1, 12, 18→1]

Â[1][0] ⊙ r̂[1]: [7×3, 14×12, 3×13, 10×10, 5×1, 12×11, 1×4, 8×14]
 = [21→4, 168→15, 39→5, 100→15, 5, 132→13, 4, 112→10]

NTT sum: [10, 19→2, 7, 22→5, 20→3, 14, 16, 11]

INTT([10,2,7,5,3,14,16,11]) = [0, 15, 3, 11, 9, 16, 3, 4]

+ e1[0] = [0,0,0,1,0,0,0,0]: u[0] = [0, 15, 3, 12, 9, 16, 3, 4] ✓
```

### Exercise 6: Implement `encapsulate`

In [ ]:
def encode_message(msg: list, q: int) -> list:
    """Encode each bit: 0 → 0,  1 → q//2."""
    # TODO: return [bit * (q // 2)  for bit in msg]
    return [... for bit in msg]

def encapsulate(A_hat, t_hat, r, e1, e2, msg, roots, inv_roots, q, k, n_inv):
    """Kyber encapsulation. Returns (u, v).

    Steps:
      1. r_ntt = [ntt(rr, roots, q)  for rr in r]
      2. A_T[i][j] = A_hat[j][i]       (transpose)
      3. ATr = matvec_ntt(A_T, r_ntt, q)
      4. u[row] = intt(ATr[row]) + e1[row]  mod q
      5. tTr[i] = sum_col( t_hat[col][i] * r_ntt[col][i] ) mod q
      6. v = intt(tTr) + e2 + encode_message(msg)  mod q
    """
    n = len(r[0])

    # Step 1: NTT of r
    # TODO: r_ntt = [ntt(rr, roots, q)  for rr in r]
    r_ntt = [... for rr in r]

    # Step 2, transpose A: A_T[i][j] = A_hat[j][i]
    # TODO: A_T = [[A_hat[j][i] for j in range(k)] for i in range(k)]
    A_T = [[... for j in range(k)] for i in range(k)]

    # Steps 3 & 4: u (provided, uses your r_ntt and A_T above)
    ATr_ntt = matvec_ntt(A_T, r_ntt, q)
    u = []
    for row in range(k):
        u_row = intt(ATr_ntt[row], inv_roots, q, n_inv)
        u.append([(u_row[i] + e1[row][i]) % q for i in range(n)])

    # Steps 5 & 6: v (provided, uses your r_ntt above)
    tTr_ntt = [sum(t_hat[col][i] * r_ntt[col][i] for col in range(k)) % q
               for i in range(n)]
    v_pre = intt(tTr_ntt, inv_roots, q, n_inv)
    m_enc = encode_message(msg, q)
    v = [(v_pre[i] + e2[i] + m_enc[i]) % q for i in range(n)]
    return u, v

u_res, v_res = encapsulate(A,[T0,T1],[R0,R1],[E1_0,E1_1],E2,MSG,ROOTS,INV_ROOTS,Q,K,N_INV)
print(f"u[0]={u_res[0]}  expected {U0}")
print(f"u[1]={u_res[1]}  expected {U1}")
print(f"v   ={v_res}  expected {V}")
assert u_res[0]==U0 and u_res[1]==U1 and v_res==V
print("\n✓ Ciphertext (u, v) matches the global constants.")


## Module 7: Decapsulation

### 7.1 Alice's computation

Alice holds the secret key `ŝ` and receives ciphertext `(u, v)`:

```
w = v − INTT(ŝ^T · NTT(u))
```

Then each coefficient of w is decoded to a bit by **circular distance**:
- `dist_to_0 = min(w[i], q − w[i])`
- `dist_to_q/2 = min(|w[i]−8|, q − |w[i]−8|)`
- **bit = 1** if `dist_to_q/2 < dist_to_0`, else **bit = 0**

### 7.2 Why the algebra works: the cancellation

Expanding v and u algebraically:
```
v = INTT(t̂^T·r̂) + e2 + m̂
 = INTT((Â·ŝ + ê)^T·r̂) + e2 + m̂
 = INTT(ŝ^T·Â^T·r̂) + INTT(ê^T·r̂) + e2 + m̂

ŝ^T·u = INTT(ŝ^T·(Â^T·r̂ + NTT(e1)))
 = INTT(ŝ^T·Â^T·r̂) + INTT(ŝ^T·NTT(e1))

w = v − ŝ^T·u
 = m̂ + INTT(ê^T·r̂) + e2 − INTT(ŝ^T·NTT(e1))
 └──────────────────────────────────────────┘
 NOISE TERMS ONLY
```

The `INTT(ŝ^T·Â^T·r̂)` terms cancel exactly.
Only noise remains, and since s, e, r, e1, e2 all have small coefficients, the noise is small.

### 7.3 Noise budget

```
i w[i] m̂[i] noise = w[i]−m̂[i] |noise| < q/4=4.25?
0 10 8 +2 2 < 4.25 ✓
1 16 0 −1 (16−17=−1) 1 < 4.25 ✓
2 8 8 0 0 < 4.25 ✓
3 0 0 0 0 < 4.25 ✓
4 9 8 +1 1 < 4.25 ✓
5 1 0 +1 1 < 4.25 ✓
6 7 8 −1 1 < 4.25 ✓
7 0 0 0 0 < 4.25 ✓
Max noise = 2. All within budget. Decryption succeeds.
```

### Exercise 7: Implement `decapsulate`

In [ ]:
def decode_bit(coeff: int, q: int) -> int:
    """Decode one coefficient to a bit.

    d0  = min(coeff, q-coeff)          distance to 0
    dq2 = min(|coeff-q//2|,            distance to q//2
               q-|coeff-q//2|)
    return 1 if dq2 < d0 else 0
    """
    d0  = min(coeff, q - coeff)
    dq2 = min(abs(coeff - q//2), q - abs(coeff - q//2))
    # TODO: return 1 if dq2 < d0 else 0
    return ...

def decapsulate(s_hat, u, v, roots, inv_roots, q, k, n_inv):
    """Kyber decapsulation. Returns decoded bit list.

    Steps:
      1. u_ntt[i] = ntt(u[i], roots, q)
      2. stu[i]   = sum_j( s_hat[j][i] * u_ntt[j][i] ) mod q
      3. stu_n    = intt(stu, inv_roots, q, n_inv)
      4. w[i]     = (v[i] - stu_n[i]) % q
      5. return [decode_bit(c, q) for c in w]
    """
    n = len(u[0])
    # TODO: u_ntt = [ntt(uu, roots, q)  for uu in u]
    u_ntt   = [... for uu in u]
    stu_ntt = [sum(s_hat[j][i]*u_ntt[j][i] for j in range(k))%q for i in range(n)]
    # TODO: stu_n = intt(stu_ntt, inv_roots, q, n_inv)
    stu_n   = ...
    # TODO: w = [(v[i] - stu_n[i]) % q  for i in range(n)]
    w       = [... for i in range(n)]
    # TODO: return [decode_bit(c, q)  for c in w]
    return [... for c in w]

result = decapsulate([S0_NTT,S1_NTT],[U0,U1],V,ROOTS,INV_ROOTS,Q,K,N_INV)
print(f"Decoded:  {result}")
print(f"Original: {MSG}")
assert result == DECODED
print("\n✓ Decapsulation matches the global constants.")


## Module 8: Full System Test

### 8.1 The complete KeyGen → Encapsulate → Decapsulate pipeline

This cell assembles everything into one run:

```
KeyGen:
 s_ntt, e_ntt = NTT(s), NTT(e)
 t_hat = A * s_ntt + e_ntt

Encapsulate:
 r_ntt = NTT(r)
 u = INTT(A^T * r_ntt) + e1
 v = INTT(t^T * r_ntt) + e2 + encode(m)

Decapsulate:
 w = v - INTT(s^T * NTT(u))
 m' = decode(w)
 assert m' == m
```

### 8.2 Why Kyber is post-quantum secure

Shor's algorithm breaks RSA and ECDSA by exploiting **periodicity** because the function `f(x) = aˣ mod N` repeats, and the Quantum Fourier Transform detects the period.

The LWE function `t = A·s + e` has **no periodicity**. The fresh noise `e` is different every time, preventing any periodic structure. The QFT finds nothing to lock onto.

```
RSA: f(x) = 7^x mod 15 → [1,7,4,13,1,7,4,13,...] period = 4 (QFT breaks this)
LWE: t = A·s + e → output changes randomly no period (QFT finds nothing)
```

**Security levels:**
- Kyber-512: 2¹¹⁸ quantum operations to break
- Kyber-768: 2¹⁷⁸ quantum operations to break
- Kyber-1024: 2²⁵⁴ quantum operations to break

### Exercise 8: Run the full pipeline and verify end-to-end

In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║   CRYSTALS-Kyber  Full System Test                       ║")
print(f"║   q={Q}  n={N}  k={K}  ETA={ETA}                                    ║")
print("╚══════════════════════════════════════════════════════════╝\n")

# KeyGen
print("KeyGen")
s_ntt_kg = [ntt(S0, ROOTS, Q), ntt(S1, ROOTS, Q)]
e_ntt_kg = [ntt(E0, ROOTS, Q), ntt(E1, ROOTS, Q)]
t_hat_kg = keygen_t(A, s_ntt_kg, e_ntt_kg, Q)
assert t_hat_kg[0]==T0 and t_hat_kg[1]==T1
print(f"  t̂[0] = {t_hat_kg[0]}")
print(f"  t̂[1] = {t_hat_kg[1]}")
print("  ✓ KeyGen OK\n")

# Encapsulation
print("Encapsulation")
print(f"  message m = {MSG}")
u_enc, v_enc = encapsulate(
    A, t_hat_kg, [R0,R1], [E1_0,E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)
assert u_enc[0]==U0 and u_enc[1]==U1 and v_enc==V
print(f"  u[0] = {u_enc[0]}")
print(f"  u[1] = {u_enc[1]}")
print(f"  v    = {v_enc}")
print("  ✓ Encapsulation OK\n")

# Decapsulation
print("Decapsulation")
m_recovered = decapsulate(s_ntt_kg, u_enc, v_enc, ROOTS, INV_ROOTS, Q, K, N_INV)
assert m_recovered == MSG
print(f"  w       = {W}")
print(f"  decoded = {m_recovered}")
print(f"  original= {MSG}")
print("  ✓ Decapsulation OK\n")

# Noise report
print("Noise Budget")
for i in range(N):
    noise = (W[i] - M_HAT[i]) % Q
    if noise > Q//2: noise -= Q
    bar = "█" * abs(noise) + "░" * (Q//4 - abs(noise))
    print(f"  coeff[{i}]  w={W[i]:2d}  m̂={M_HAT[i]}  noise={noise:+2d}  [{bar}]  < q/4={Q//4}")

print("\n╔══════════════════════════════════════════════════════════╗")
print("║    FULL SYSTEM TEST PASSED                              ║")
print("║  KeyGen → Encapsulate → Decapsulate  ✓                   ║")
print("╚══════════════════════════════════════════════════════════╝")